# Modelo 3 — Clasificador Binario de Alerta de Precios
**Proyecto:** food-price-forecasting-peru | Big Data Analytics UNMSM 2025-II

**Pregunta:** El precio de este alimento va a subir mas del 10% en las proximas dos semanas?

**Algoritmos:** Random Forest Classifier (principal) vs GBT Classifier (comparacion)

**Flujo:** Cargar Plata (matriz_aprendizaje_forecasting) → Verificar desbalance → Limpiar nulos → VectorAssembler → Split cronologico → Entrenar → MLflow → Zona Oro

In [1]:
# Instalacion (ejecutar una vez por sesion)
!pip install -q pyspark mlflow google-cloud-storage

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [2]:
# Variables de configuracion
BUCKET_NAME = "food-price-peru-2025-01"
PROJECT_ID  = "proyecto-food-price"

PATH_PLATA = f"gs://{BUCKET_NAME}/plata/precios_clima_integrado/matriz_aprendizaje_forecasting/"
PATH_ORO   = f"gs://{BUCKET_NAME}/oro/alerta_binaria_productos/"

print(f"Entrada : {PATH_PLATA}")
print(f"Salida  : {PATH_ORO}")

Entrada : gs://food-price-peru-2025-01/plata/precios_clima_integrado/matriz_aprendizaje_forecasting/
Salida  : gs://food-price-peru-2025-01/oro/alerta_binaria_productos/


In [4]:
# Autenticacion GCP e inicializacion de Spark
import os
from google.colab import auth
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

auth.authenticate_user()
!gcloud config set project {PROJECT_ID}

jar_name = "gcs-connector.jar"
jar_path = os.path.abspath(jar_name)
if not os.path.exists(jar_path):
    !wget -q https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.14/gcs-connector-hadoop3-2.2.14-shaded.jar -O {jar_name}

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {jar_path} pyspark-shell"

spark = (SparkSession.builder
    .appName("D08-Modelo3-Clasification")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.driver.extraClassPath", jar_path)
    .config("spark.executor.extraClassPath", jar_path)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado. Version:", spark.version)

Updated property [core/project].
Spark iniciado. Version: 4.0.3


In [5]:
# Cargar data desde Zona Plata
df_plata = spark.read.parquet(PATH_PLATA)

print("Schema:")
df_plata.printSchema()
print(f"Total filas: {df_plata.count()}")
print("Productos:")
df_plata.groupBy("producto").count().orderBy("producto").show(truncate=False)

Schema:
root
 |-- producto: string (nullable = true)
 |-- region_clima_origen: string (nullable = true)
 |-- anio: long (nullable = true)
 |-- semana: long (nullable = true)
 |-- precio_promedio: double (nullable = true)
 |-- precio_min: double (nullable = true)
 |-- precio_max: double (nullable = true)
 |-- precio_std: double (nullable = true)
 |-- n_obs: long (nullable = true)
 |-- mercado_sisap: string (nullable = true)
 |-- distrito_lima: string (nullable = true)
 |-- ubigeo_inei: string (nullable = true)
 |-- temp_avg: double (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_min: double (nullable = true)
 |-- precip_total: double (nullable = true)
 |-- humedad_avg: double (nullable = true)
 |-- dias_helada: double (nullable = true)
 |-- dias_lluvia_intensa: double (nullable = true)
 |-- n_obs_clima: double (nullable = true)
 |-- precio_lag_1: double (nullable = true)
 |-- precio_lag_2: double (nullable = true)
 |-- precio_lag_4: double (nullable = true)
 |-- prec

In [6]:
# Verificar distribucion de la variable objetivo (desbalance de clases)

dist = df_plata.groupBy("target_alerta_10").count().orderBy("target_alerta_10")
dist.show()

total = df_plata.count()
pct_alerta = df_plata.filter(F.col("target_alerta_10") == 1).count() / total * 100
print(f"Porcentaje de semanas con alerta (subida > 10% en 2 semanas): {pct_alerta:.2f}%")
print(f"Porcentaje de semanas sin alerta: {100 - pct_alerta:.2f}%")
print("\nDesbalance moderado (no extremo). Se usara AUC-ROC y F1-Score como")
print("metricas principales en lugar de accuracy, que seria enganoso aqui.")

+----------------+-----+
|target_alerta_10|count|
+----------------+-----+
|               0| 3955|
|               1|  742|
+----------------+-----+

Porcentaje de semanas con alerta (subida > 10% en 2 semanas): 15.80%
Porcentaje de semanas sin alerta: 84.20%

Desbalance moderado (no extremo). Se usara AUC-ROC y F1-Score como
metricas principales en lugar de accuracy, que seria enganoso aqui.


In [7]:
# Diagnostico de nulos y limpieza
# Se elimina "dias_helada" de cols_clima porque tiene varianza 0 en el subset agricola

cols_clima = ["temp_avg", "precip_total", "humedad_avg", "dias_lluvia_intensa"]

NO_AGRICOLAS = [
    "Fideos Molitalia Spaghetti / Tallarin",
    "Huevos Rosados",
    "Leche Fresca Gloria En Bolsa ( 12 Bolsas De Litro)",
    "Pollo Vivo"
]

df_model = df_plata.filter(F.col("target_precio_2s").isNotNull())
df_model = df_model.filter(~F.col("producto").isin(NO_AGRICOLAS))
df_model = df_model.dropna(subset=cols_clima)

print(f"Filas originales: {df_plata.count()}")
print(f"Filas tras excluir no agricolas: {df_model.count()}")
df_model.groupBy("producto").count().orderBy("producto").show(truncate=False)

Filas originales: 4697
Filas tras excluir no agricolas: 2341
+--------------------+-----+
|producto            |count|
+--------------------+-----+
|Ajo Criollo O Napuri|398  |
|Camote Morado       |381  |
|Papa Amarilla       |383  |
|Papaya              |399  |
|Platano Bellaco     |399  |
|Zanahoria           |381  |
+--------------------+-----+



In [9]:
# Elección de features
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder

window_prod = Window.partitionBy("producto").orderBy("anio", "semana")

# Solo se conservan features NO redundantes entre si.
FEATURE_COLS_NUMERICAS = [
    "precio_lag_1", "precio_lag_2", "precio_lag_4",
    "precio_std_4", "variacion_1",
    "temp_avg", "precip_total", "humedad_avg", "dias_lluvia_intensa",
    "semana", "mes", "trimestre"
]

# Codificar "producto" como feature — esto SI aporta señal real:
indexer_producto = StringIndexer(inputCol="producto", outputCol="producto_idx", handleInvalid="keep")
encoder_producto = OneHotEncoder(inputCols=["producto_idx"], outputCols=["producto_ohe"])

df_model_idx = indexer_producto.fit(df_model).transform(df_model)
df_model_ohe = encoder_producto.fit(df_model_idx).transform(df_model_idx)

print(f"Total features numericas: {len(FEATURE_COLS_NUMERICAS)}")
print("Se agrega ademas el vector one-hot de producto (6 categorias)")

Total features numericas: 12
Se agrega ademas el vector one-hot de producto (6 categorias)


In [11]:
# Ensamblado del vector final:
from pyspark.ml.feature import VectorAssembler

FEATURE_COLS_FINAL = FEATURE_COLS_NUMERICAS + ["producto_ohe"]

assembler = VectorAssembler(inputCols=FEATURE_COLS_FINAL, outputCol="features", handleInvalid="skip")

df_features = assembler.transform(df_model_ohe).select(
    "producto", "anio", "semana", "features",
    F.col("target_alerta_10").cast("double").alias("label")
)

print(f"Features numericas: {len(FEATURE_COLS_NUMERICAS)} + vector producto")
df_features.select("producto", "anio", "semana", "label").show(5)

Features numericas: 12 + vector producto
+--------------------+----+------+-----+
|            producto|anio|semana|label|
+--------------------+----+------+-----+
|Ajo Criollo O Napuri|2017|     1|  0.0|
|Ajo Criollo O Napuri|2017|     2|  0.0|
|Ajo Criollo O Napuri|2017|     3|  1.0|
|Ajo Criollo O Napuri|2017|     4|  0.0|
|Ajo Criollo O Napuri|2017|     5|  0.0|
+--------------------+----+------+-----+
only showing top 5 rows


In [12]:
# Split cronologico train / test

df_train = df_features.filter(F.col("anio") <= 2022)
df_test  = df_features.filter(F.col("anio") >= 2023)

print(f"Registros entrenamiento: {df_train.count()}")
print(f"Registros prueba: {df_test.count()}")

# Limpiamos cache anterior y guardamos el nuevo set
df_train.unpersist()
df_test.unpersist()
df_train.cache()
df_test.cache()

Registros entrenamiento: 1742
Registros prueba: 599


DataFrame[producto: string, anio: bigint, semana: bigint, features: vector, label: double]

In [13]:
# Verificar estado de MLflow
import mlflow

# Solo nos aseguramos de que el experimento sea el correcto
experiment_name = "D08-Modelo3-ClasificacionBinaria"
mlflow.set_experiment(experiment_name)

print(f"MLflow listo. Registrando en: {experiment_name}")

2026/07/11 00:54:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/11 00:54:55 INFO mlflow.store.db.utils: Updating database tables
2026/07/11 00:54:59 INFO mlflow.tracking.fluent: Experiment with name 'D08-Modelo3-ClasificacionBinaria' does not exist. Creating a new experiment.


MLflow listo. Registrando en: D08-Modelo3-ClasificacionBinaria


In [14]:
# Random Forest

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

count_0 = df_train.filter(F.col("label") == 0).count()
count_1 = df_train.filter(F.col("label") == 1).count()
total_rows = df_train.count()
weight_0 = total_rows / (2.0 * count_0)
weight_1 = total_rows / (2.0 * count_1)
df_train = df_train.withColumn("classWeight",
    F.when(F.col("label") == 1, weight_1).otherwise(weight_0))

eval_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

resultados_modelos = []

with mlflow.start_run(run_name="RF_Corregido_Depth6"):
    rf = RandomForestClassifier(
        labelCol="label", featuresCol="features", weightCol="classWeight", seed=42
    )

    paramGrid = (ParamGridBuilder()
        .addGrid(rf.numTrees, [300])
        .addGrid(rf.maxDepth, [4, 6])
        .build())

    cv = CrossValidator(estimator=rf, estimatorParamMaps=paramGrid, evaluator=eval_auc, numFolds=3)
    modelo_cv = cv.fit(df_train)
    modelo_rf = modelo_cv.bestModel

    pred_final = modelo_rf.transform(df_test)
    # Se reporta al umbral estandar 0.5 — no se optimiza sobre df_test
    # porque eso seria una fuga de informacion del test set hacia el modelo.
    auc_final = eval_auc.evaluate(pred_final)
    f1_final  = eval_f1.evaluate(pred_final)

    mlflow.log_param("algoritmo", "RandomForest")
    mlflow.log_param("maxDepth_probado", "4,6")
    mlflow.log_param("num_features", len(FEATURE_COLS_FINAL))
    mlflow.log_metric("auc_roc", auc_final)
    mlflow.log_metric("f1", f1_final)

    resultados_modelos.append({"nombre": "RandomForest", "modelo": modelo_rf,
                                "predicciones": pred_final, "auc": auc_final, "f1": f1_final})
    print(f"RandomForest → AUC: {auc_final:.4f} | F1: {f1_final:.4f}")

RandomForest → AUC: 0.6651 | F1: 0.6628


In [16]:
# GBT Classifier

from pyspark.ml.classification import GBTClassifier

with mlflow.start_run(run_name="GBTClassifier"):
    gbt = GBTClassifier(
        labelCol="label", featuresCol="features",
        maxIter=100, maxDepth=5, seed=42
    )
    modelo_gbt = gbt.fit(df_train)
    pred_gbt   = modelo_gbt.transform(df_test)

    auc_gbt = eval_auc.evaluate(pred_gbt)
    f1_gbt  = eval_f1.evaluate(pred_gbt)

    mlflow.log_param("algoritmo", "GBTClassifier")
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("n_features", len(FEATURE_COLS_FINAL))
    mlflow.log_metric("auc_roc", auc_gbt)
    mlflow.log_metric("f1_score", f1_gbt)

    resultados_modelos.append({"nombre": "GBT", "modelo": modelo_gbt,
                                "predicciones": pred_gbt, "auc": auc_gbt, "f1": f1_gbt})
    print(f"GBT -> AUC-ROC: {auc_gbt:.4f} | F1-Score: {f1_gbt:.4f}")

GBT -> AUC-ROC: 0.6707 | F1-Score: 0.6883


In [18]:
# Regresion Logistica con peso balanceado
from pyspark.ml.classification import LogisticRegression

with mlflow.start_run(run_name="LogisticRegression_Baseline"):
    lr = LogisticRegression(
        labelCol="label", featuresCol="features", weightCol="classWeight",
        maxIter=100, regParam=0.01
    )
    modelo_lr = lr.fit(df_train)
    pred_lr = modelo_lr.transform(df_test)

    auc_lr = eval_auc.evaluate(pred_lr)
    f1_lr  = eval_f1.evaluate(pred_lr)

    mlflow.log_param("algoritmo", "LogisticRegression")
    mlflow.log_metric("auc_roc", auc_lr)
    mlflow.log_metric("f1", f1_lr)

    # Evitar duplicados en la lista de resultados
    resultados_modelos = [r for r in resultados_modelos if r['nombre'] != 'LogisticRegression']
    resultados_modelos.append({"nombre": "LogisticRegression", "modelo": modelo_lr,
                                "predicciones": pred_lr, "auc": auc_lr, "f1": f1_lr})
    print(f"LogisticRegression → AUC: {auc_lr:.4f} | F1: {f1_lr:.4f}")

LogisticRegression → AUC: 0.6324 | F1: 0.5811


In [21]:
# Actualizar resultados con los modelos optimizados y buscar el mejor threshold
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

eval_f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')

# Reemplazamos en la lista de resultados con las versiones mas potentes
pred_rf_opt = best_rf.transform(df_test)
pred_gbt_opt = best_gbt.transform(df_test)

resultados_modelos = [
    {'nombre': 'RandomForest (Opt)', 'modelo': best_rf, 'predicciones': pred_rf_opt, 'auc': res_rf, 'f1': eval_f1.evaluate(pred_rf_opt)},
    {'nombre': 'GBT (Opt)', 'modelo': best_gbt, 'predicciones': pred_gbt_opt, 'auc': res_gbt, 'f1': eval_f1.evaluate(pred_gbt_opt)},
    resultados_modelos[2] # Mantenemos la Regresion Logistica inicial
]

ganador = max(resultados_modelos, key=lambda x: x['auc'])

print('='*60)
print('COMPARATIVA ACTUALIZADA (Modelos Optimizados)')
print('='*60)
for r in resultados_modelos:
    print(f"  {r['nombre']:<25}{r['auc']:<15.4f}{r['f1']:<15.4f}")
print('-'*60)
print(f"GANADOR DEFINITIVO: {ganador['nombre']} | AUC: {ganador['auc']:.4f}")

COMPARATIVA ACTUALIZADA (Modelos Optimizados)
  RandomForest (Opt)       0.6756         0.7135         
  GBT (Opt)                0.6821         0.6720         
  LogisticRegression       0.6324         0.5811         
------------------------------------------------------------
GANADOR DEFINITIVO: GBT (Opt) | AUC: 0.6821


In [24]:
# Matriz de confusion e importancia de features del ganador
pred_ganador = ganador["predicciones"]

print("Matriz de confusion (test set):")
pred_ganador.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

print("\nTop 10 features por importancia:")
# Se corrige FEATURE_COLS por FEATURE_COLS_FINAL que es la variable definida anteriormente
importancias = list(zip(FEATURE_COLS_FINAL, ganador["modelo"].featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)
for feat, imp in importancias[:10]:
    print(f"  {feat:<25} {imp:.4f}")

Matriz de confusion (test set):
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  421|
|  0.0|       1.0|   35|
|  1.0|       0.0|  129|
|  1.0|       1.0|   14|
+-----+----------+-----+


Top 10 features por importancia:
  semana                    0.1135
  temp_avg                  0.1080
  precio_lag_1              0.1034
  precio_lag_4              0.1029
  precio_std_4              0.0862
  variacion_1               0.0852
  precio_lag_2              0.0763
  humedad_avg               0.0726
  precip_total              0.0563
  mes                       0.0409


In [26]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Definir el evaluador para AUC
eval_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability", metricName="areaUnderROC")

# 2. Configurar el modelo GBT
gbt_cv = GBTClassifier(labelCol="label", featuresCol="features", seed=42)

# 3. Crear un grid (aunque sea pequeño para validar el rendimiento promedio)
grid_cv = (ParamGridBuilder()
    .addGrid(gbt_cv.maxIter, [50, 100])
    .addGrid(gbt_cv.maxDepth, [3, 5])
    .build())

# 4. Configurar CrossValidator (3 Folds es estándar para este volumen de data)
cv = CrossValidator(
    estimator=gbt_cv,
    estimatorParamMaps=grid_cv,
    evaluator=eval_auc,
    numFolds=3,
    parallelism=2
)

print("Ejecutando Validación Cruzada para determinar el techo de performance...")
modelo_cv = cv.fit(df_train)

# 5. Resultados
auc_promedio = max(modelo_cv.avgMetrics)
print(f"\nRESULTADO DE VALIDACIÓN CRUZADA:")
print(f"AUC-ROC Promedio (CV): {auc_promedio:.4f}")

Ejecutando Validación Cruzada para determinar el techo de performance...

RESULTADO DE VALIDACIÓN CRUZADA:
AUC-ROC Promedio (CV): 0.7122


In [27]:
# Guardar predicciones finales en Zona Oro (GCS)
from pyspark.ml.functions import vector_to_array

# Extraemos la probabilidad de la clase 1 (Alerta)
df_oro = (pred_ganador
    .withColumn("prob_alerta", vector_to_array(F.col("probability"))[1])
    .select(
        "producto", "anio", "semana",
        F.col("label").alias("alerta_real"),
        F.col("prediction").alias("alerta_predicha"),
        "prob_alerta"
    )
    .withColumn("modelo", F.lit(ganador["nombre"]))
    .withColumn("auc_test", F.lit(round(ganador["auc"], 4)))
    .withColumn("auc_cv_potencial", F.lit(0.7122)) # Referencia de la validación cruzada
)

# Escritura en GCS
df_oro.write.mode("overwrite").parquet(PATH_ORO)

print(f"--- Datos persistidos con éxito ---")
print(f"Ruta: {PATH_ORO}")

# Verificación rápida
df_check = spark.read.parquet(PATH_ORO)
print(f"Total registros en Oro: {df_check.count()}")
df_check.select("producto", "alerta_real", "alerta_predicha", "prob_alerta").show(5)

--- Datos persistidos con éxito ---
Ruta: gs://food-price-peru-2025-01/oro/alerta_binaria_productos/
Total registros en Oro: 599
+--------------------+-----------+---------------+-------------------+
|            producto|alerta_real|alerta_predicha|        prob_alerta|
+--------------------+-----------+---------------+-------------------+
|Ajo Criollo O Napuri|        1.0|            0.0|0.38429843987958123|
|Ajo Criollo O Napuri|        1.0|            0.0| 0.4575683378234173|
|Ajo Criollo O Napuri|        0.0|            0.0|0.41497443096136233|
|Ajo Criollo O Napuri|        0.0|            0.0|0.12580795700231717|
|Ajo Criollo O Napuri|        0.0|            0.0|0.07375523807450235|
+--------------------+-----------+---------------+-------------------+
only showing top 5 rows


In [29]:
# CONCLUSIÓN Y DESCARGA DE EXPERIMENTOS
from google.colab import files
import shutil
import os

print("="*60)
print("RESUMEN EJECUTIVO - MODELO GANADOR")
print("="*60)
print(f"Algoritmo : {ganador['nombre']}")
print(f"AUC-ROC   : {ganador['auc']:.4f} (Potencial CV: 0.7122)")
print(f"F1-Score  : {ganador['f1']:.4f}")
print("\nJustificación rápida:")
print("- GBT superó a RF y LR capturando mejor las no linealidades del clima.")
print("- La volatilidad de precios agrícolas limita el techo predictivo a ~0.71.")
print("- Los datos están listos en Zona Oro para el Dashboard.")
print("="*60)

# Proceso de descarga de MLflow (Base de datos y carpetas)
print("\nPreparando archivos de experimentos para descarga...")

# Creamos una carpeta temporal para juntar todo lo relevante de MLflow
temp_export = "respaldo_mlflow"
if not os.path.exists(temp_export): os.makedirs(temp_export)

if os.path.exists("mlflow.db"): shutil.copy("mlflow.db", temp_export)
if os.path.exists("mlruns"): shutil.copytree("mlruns", f"{temp_export}/mlruns", dirs_exist_ok=True)

shutil.make_archive('experimentos_clasificacion', 'zip', temp_export)
files.download('experimentos_clasificacion.zip')

print("¡Listo! El archivo experimentos_clasificacion.zip se está descargando.")

RESUMEN EJECUTIVO - MODELO GANADOR
Algoritmo : GBT (Opt)
AUC-ROC   : 0.6821 (Potencial CV: 0.7122)
F1-Score  : 0.6720

Justificación rápida:
- GBT superó a RF y LR capturando mejor las no linealidades del clima.
- La volatilidad de precios agrícolas limita el techo predictivo a ~0.71.
- Los datos están listos en Zona Oro para el Dashboard.

Preparando archivos de experimentos para descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

¡Listo! El archivo experimentos_clasificacion.zip se está descargando.
